To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/new/standby) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    pass

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Gemma 3 1B Instruct`, and set parameters

In [ ]:
from unsloth import FastModel
import torch
max_seq_length = 1024

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

### Data Prep
<a name="Data"></a>

We use our custom training data for Michi (sarcastic desktop cat). Upload your `training_data.jsonl` or fetch it from GitHub.

In [ ]:
import json
import random
from datasets import Dataset

# Option A: Upload from local (use Colab file upload)
# Option B: Fetch from GitHub (uncomment and set your URL)
import requests
url = "https://raw.githubusercontent.com/EmilianoDorantes/desktopPet/master/training_data.jsonl"
response = requests.get(url)
lines = response.content.decode("utf-8-sig").strip().split("\n")

# Parse JSONL
records = [json.loads(line) for line in lines]
print(f"Loaded {len(records)} examples")

# Convert to prompt/answer format for GRPO
# prompt = [system_msg, user_msg]  (all except assistant)
# answer = assistant content
def convert(record):
    msgs = record["messages"]
    prompt = msgs[:-1]  # system + user
    answer = msgs[-1]["content"]
    return {"prompt": prompt, "answer": answer}

dataset = Dataset.from_list([convert(r) for r in records])
dataset

Let's look at the first row:

In [ ]:
dataset[0]["prompt"]

In [ ]:
dataset[0]["answer"]

### Reward Functions

Define rewards to shape Michi's sarcastic personality:
- **length_reward**: Keep responses short (1-2 sentences, ~20-200 chars)
- **no_emoji_reward**: Penalize emoji use
- **tone_reward**: Reward sarcastic/acidic tone, penalize helpful/positive language
- **gold_reward**: Reward similarity to the gold training response

In [ ]:
import re
import unicodedata

def is_emoji(char):
    try:
        return unicodedata.category(char) == 'So'
    except:
        return False

# Palabras clave sarcásticas/ácidas que Michi debe priorizar
SARCASTIC_WORDS = {
    'lastima', 'suerte', 'seguro', 'claro', 'obvio', 'tranquilo',
    'viste', 'sabes', 'creo', 'parece', 'anda', 'dale', 'bueno',
    'total', 'tanto', 'nunca', 'siempre', 'jamas', 'peor', 'mejor',
    'felicidades', 'genial', 'maravilloso', 'excelente', 'espectacular',
    'interesante', 'ajá', 'así', 'humano', 'mira', 'obviamente'
}

# Palabras serviciales/positivas que Michi debe EVITAR a toda costa
HELPFUL_WORDS = {
    'ayudarte', 'gustaria', 'encantaria', 'feliz', 'contento',
    'complacido', 'sugiero', 'recomiendo', 'podrias', 'deberias',
    'permíteme', 'claro que si', 'por supuesto', 'con gusto',
    'ayuda', 'servirle', 'asistirle', 'puedo', 'hacer', 'hoy'
}

def length_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        length = len(response)
        
        # Penalizaciones extremas por textos largos o nulos
        if length < 8:
            scores.append(-2.0)
        elif length > 120:
            scores.append(-2.5)  # Castigo fuerte por hablar demasiado
        elif 15 <= length <= 80:
            scores.append(2.0)   # Rango ideal para una o dos frases cortas
        else:
            scores.append(0.5)
    return scores


def no_emoji_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        emoji_count = sum(1 for c in response if is_emoji(c))
        if emoji_count == 0:
            scores.append(1.0)
        else:
            scores.append(-1.5 * emoji_count) # Penalización ligeramente más severa
    return scores


def tone_reward(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Limpieza básica para extraer palabras bien definidas
        words = set(re.findall(r'[a-zA-Záéíóúñü]+', response.lower()))
        
        sarcastic_hits = len(words & SARCASTIC_WORDS)
        helpful_hits = len(words & HELPFUL_WORDS)
        
        # Premiamos el sarcasmo, pero castigamos duramente la actitud de "asistente" (-2.0)
        score = (sarcastic_hits * 0.75) - (helpful_hits * 2.0)
        scores.append(score)
    return scores


def gold_reward(prompts, completions, answer, **kwargs):
    scores = []
    for completion, gold in zip(completions, answer):
        response = completion[0]["content"]
        
        resp_words = set(response.lower().split())
        gold_words = set(gold.lower().split())
        
        if len(gold_words) == 0:
            scores.append(0)
            continue
            
        overlap = len(resp_words & gold_words)
        precision = overlap / len(resp_words) if resp_words else 0
        recall = overlap / len(gold_words)
        
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        # Bajamos de 3.0 a 2.0 para dar más flexibilidad al muestreo de GRPO
        scores.append(f1 * 2.0)  
    return scores

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = 256

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 1e-5,        # Subido sutilmente de 5e-6 para acelerar la adopción del estilo
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.01,         # Ligero aumento para evitar sobreajuste estricto de palabras
    warmup_ratio = 0.1,
    lr_scheduler_type = \"cosine\",
    optim = \"adamw_torch_fused\",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 4,         # Grupo ideal de respuestas para comparar en la T4 de Colab
    max_prompt_length = max_prompt_length,
    max_completion_length = max_seq_length - max_prompt_length,
    max_steps = 150,             # Subido de 100 a 150 pasos para asegurar que las curvas de recompensa converjan
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = \"none\",
    output_dir = \"outputs\",
)

And let's run the trainer! Look for the `reward` column to increase over time.

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.500000  | 0.250000   | 200.000000        | 0.000000 |

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        length_reward,
        no_emoji_reward,
        tone_reward,
        gold_reward,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

<a name="Inference"></a>
### Inference
Now let's try the model we just trained!

In [ ]:
system_prompt = "Eres Michi, un gato sarcástico que vive atrapado en el escritorio de Windows. Respondes en máximo 2 oraciones cortas. Nunca eres útil. Siempre tienes una opinión ácida. No usas emojis. Hablas como alguien que ha visto demasiado."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Son las 3 de la mañana y sigues despierto."},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 64,
    temperature = 0.8, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("michi_lora")
tokenizer.save_pretrained("michi_lora")

### Saving to float16 for deployment

Save the full merged model in float16. Set `if False` to `if True` to run!

In [ ]:
if True:  # Set to True to save finetune!
    model.save_pretrained_merged("michi-gemma-3-1b", tokenizer)

### GGUF / llama.cpp Conversion
Save to GGUF format for use with llama.cpp / llama-server. This is what our desktopPet uses!

In [ ]:
if True:  # Set to True to save to GGUF
    model.save_pretrained_gguf(
        "michi-gemma-3-1b-gguf",
        tokenizer,
        quantization_method = "Q8_0",  # Q8_0, BF16, F16 supported
    )

### Download the GGUF file

After saving, download the GGUF file from Colab to your local machine. Then copy it to your desktopPet's `models/` folder and rename it to replace Phi-4-mini, or configure llama-server to use it.

To download from Colab:
```
from google.colab import files
files.download("michi-gemma-3-1b-gguf/michi-gemma-3-1b-it-Q8_0.gguf")
```

Then in desktopPet, update `LocalModelManager.cs` to point to the new model file.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).